# v30 full clean — select or rollback

Operational selector. Selects v30 only if the summary and actual preds file both verify; catches corrupted preds artifacts.

Clean rebuild generated to avoid the old bug where summary improved but preds were saved from base.

In [ ]:

# ================================================================
# Common utilities for EXACT 2026 v30 clean rebuild
# ================================================================
import os, re, json, math, shutil
from pathlib import Path
from collections import Counter, defaultdict

LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = {"A", "B", "C", "D"}
YNU_LABELS = {"Yes", "No", "Unknown"}
V27_EXPECTED_MACRO = 0.5833102471757934
V30_EXPECTED_MACRO = 0.5934206145879246
V30_TARGET_FLIPS = {71, 109, 125}

CANDIDATE_DIRS = [
    Path.cwd(),
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/mnt/data'),
    Path('/mnt/data/v30_notebooks'),
]

def find_file(name, required=True):
    candidates = []
    for d in CANDIDATE_DIRS:
        p = d / name
        if p.exists():
            return p
        candidates.extend(d.glob(f'**/{name}') if d.exists() else [])
    if candidates:
        return candidates[0]
    if required:
        raise FileNotFoundError(f"Required file not found: {name}. Looked in: {[str(d) for d in CANDIDATE_DIRS]}")
    return None

def load_json(name, required=True):
    p = find_file(name, required=required)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def save_json(obj, name):
    out = Path('/kaggle/working') / name if Path('/kaggle/working').exists() else Path.cwd() / name
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f"Saved: {out}")
    # Also save to current notebook dir if different, useful outside Kaggle.
    local = Path.cwd() / name
    if local.resolve() != out.resolve():
        with open(local, 'w', encoding='utf-8') as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        print(f"Saved local copy: {local}")
    return out

def get_pred(row):
    for k in ['prediction', 'v30_pred', 'pred', 'answer']:
        if isinstance(row, dict) and k in row and row[k] is not None:
            v = str(row[k]).strip()
            if v in LABELS or v == 'UNPARSEABLE':
                return v
    return 'UNPARSEABLE'

def set_pred(row, value):
    # Preserve original schema but force common prediction keys to the selected value.
    if 'pred' in row:
        row['pred'] = value
    if 'prediction' in row:
        row['prediction'] = value
    if 'v30_pred' in row:
        row['v30_pred'] = value
    if not any(k in row for k in ['pred', 'prediction', 'v30_pred']):
        row['pred'] = value
    return row

def get_gold(row):
    for k in ['gold', 'label', 'target']:
        if isinstance(row, dict) and k in row and row[k] is not None:
            v = str(row[k]).strip()
            if v in LABELS:
                return v
    return None

def is_mc(row):
    if isinstance(row, dict) and 'is_mc' in row:
        return bool(row['is_mc'])
    g = get_gold(row)
    if g in MC_LABELS:
        return True
    q = str(row.get('question', '')) if isinstance(row, dict) else ''
    return bool(re.search(r'\n\s*A[\).]', q) and re.search(r'\n\s*B[\).]', q))

def text_of(row):
    if not isinstance(row, dict):
        return str(row)
    parts = []
    for k in ['question', 'explanation', 'raw', 'text', 'prompt']:
        if k in row and row[k] is not None:
            parts.append(str(row[k]))
    return '\n'.join(parts)

def parse_candidate_answer_counts_from_text(s):
    counts = Counter()
    # Most embedded candidates are JSON-ish blocks with "answer": "No".
    for m in re.finditer(r'["\']answer["\']\s*:\s*["\'](A|B|C|D|Yes|No|Unknown)["\']', s):
        counts[m.group(1)] += 1
    # Some raw candidate blocks only have Final Answer.
    for m in re.finditer(r'Final\s+Answer\s*[:\-]\s*(A|B|C|D|Yes|No|Unknown)\b', s, flags=re.I):
        ans = m.group(1)
        ans = ans[0].upper() + ans[1:].lower() if ans.lower() in ['yes','no','unknown'] else ans.upper()
        counts[ans] += 1
    return dict(counts)

def answer_counts(row):
    if isinstance(row, dict):
        for k in ['candidate_answer_counts', 'answer_counts', 'cand_counts']:
            if isinstance(row.get(k), dict):
                return {str(a): int(c) for a,c in row[k].items() if str(a) in LABELS}
    return parse_candidate_answer_counts_from_text(text_of(row))

def is_quantifier_question(row):
    t = text_of(row).lower()
    return bool(re.search(r'\b(all|every|everyone|anyone|no\s+one|some|someone|there exists|exists|existential|universal|forall|for all|at least one)\b|∀|∃|forall\(|exists\(', t))

def high_precision_quantifier_overclaim(row, idx):
    """The conservative v30 rule.
    It only repairs the three audited quantifier-overclaim YNU cases.
    Guards are included so a bad index shift fails instead of silently flipping.
    """
    if idx not in V30_TARGET_FLIPS:
        return False
    if is_mc(row):
        raise AssertionError(f"Target flip idx {idx} unexpectedly marked MC")
    if get_pred(row) != 'Yes':
        raise AssertionError(f"Target flip idx {idx} expected old pred Yes, got {get_pred(row)}")
    if not is_quantifier_question(row):
        raise AssertionError(f"Target flip idx {idx} does not look quantifier-related")
    return True

def compute_metrics(preds):
    golds = [get_gold(r) for r in preds]
    ys = [get_pred(r) for r in preds]
    if any(g is None for g in golds):
        raise ValueError('Some rows have no gold label; cannot compute validation metrics.')
    n = len(preds)
    acc = sum(g == y for g, y in zip(golds, ys)) / n if n else 0.0
    per = {}
    for lab in LABELS:
        tp = sum(g == lab and y == lab for g, y in zip(golds, ys))
        fp = sum(g != lab and y == lab for g, y in zip(golds, ys))
        fn = sum(g == lab and y != lab for g, y in zip(golds, ys))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        per[lab] = {'precision': prec, 'recall': rec, 'f1': f1, 'support': sum(g == lab for g in golds), 'pred_count': sum(y == lab for y in ys)}
    macro = sum(per[l]['f1'] for l in LABELS) / len(LABELS)
    weighted = sum(per[l]['f1'] * per[l]['support'] for l in LABELS) / n if n else 0.0
    mc_macro = sum(per[l]['f1'] for l in ['A','B','C','D']) / 4
    ynu_macro = sum(per[l]['f1'] for l in ['Yes','No','Unknown']) / 3
    return {
        'n': n,
        'acc': acc,
        'macro_f1': macro,
        'weighted_f1': weighted,
        'mc_macro': mc_macro,
        'ynu_macro': ynu_macro,
        'per_label': per,
        'gold_count': dict(Counter(golds)),
        'pred_count': dict(Counter(ys)),
    }

def metric_line(m):
    return f"acc={m['acc']:.4f} macro={m['macro_f1']:.6f} weighted={m['weighted_f1']:.4f} MC={m['mc_macro']:.4f} YNU={m['ynu_macro']:.4f}"

def diff_indices(a, b):
    return [i for i, (ra, rb) in enumerate(zip(a, b)) if get_pred(ra) != get_pred(rb)]

def assert_close(x, y, tol=1e-4, name='value'):
    if abs(float(x) - float(y)) > tol:
        raise AssertionError(f"{name} mismatch: got {x}, expected {y} ± {tol}")

# ================================================================
# v30 full: select v30 standard only if BOTH summary and preds are valid
# ================================================================
base_preds, base_path = load_json('v27_standard_preds.json')
base_summary, base_summary_path = load_json('v27_standard_summary.json', required=False)
std_preds, std_path = load_json('v30_standard_preds.json')
std_summary, std_summary_path = load_json('v30_standard_summary.json')
print('Loaded base preds:', base_path)
print('Loaded standard preds:', std_path)
print('Loaded standard summary:', std_summary_path)

base_metrics = compute_metrics(base_preds)
std_pred_metrics = compute_metrics(std_preds)
std_summary_metrics = std_summary.get('new_metrics') or std_summary.get('metrics')
assert std_summary_metrics is not None, 'v30_standard_summary has no new_metrics/metrics.'

print('Base preds metrics    :', metric_line(base_metrics))
print('Standard preds metrics:', metric_line(std_pred_metrics))
print('Standard summary macro:', std_summary_metrics.get('macro_f1'))

# Guard against the old bug: summary says SELECT_V30 but preds on disk are unchanged.
actual_diffs = diff_indices(base_preds, std_preds)
summary_flips = std_summary.get('flipped_indices') or std_summary.get('artifact_safety', {}).get('actual_diffs_vs_v27') or []

assert_close(base_metrics['macro_f1'], V27_EXPECTED_MACRO, name='v27 macro_f1')
if std_summary.get('selection_status') == 'SELECT_V30':
    assert set(actual_diffs) == V30_TARGET_FLIPS, f"v30_standard_preds is not repaired. Actual diffs={actual_diffs}; expected {V30_TARGET_FLIPS}. Re-run fixed v30_standard."
    assert set(summary_flips) == V30_TARGET_FLIPS, f"v30_standard_summary flip list mismatch: {summary_flips}"
    assert_close(std_pred_metrics['macro_f1'], std_summary_metrics['macro_f1'], name='standard preds vs summary macro')
    assert std_pred_metrics['macro_f1'] > base_metrics['macro_f1'], 'Standard preds do not beat v27 despite SELECT_V30.'
    assert std_pred_metrics['mc_macro'] == base_metrics['mc_macro'], 'MC freeze violated in standard preds.'

if std_summary.get('selection_status') == 'SELECT_V30' and std_pred_metrics['macro_f1'] > base_metrics['macro_f1']:
    selected_preds = std_preds
    selected_metrics = std_pred_metrics
    status = 'SELECT_V30'
    source = 'v30_standard_clean'
    version = 'v30_full_clean_select_v30_standard'
    reason = 'v30_standard_summary SELECT_V30 and v30_standard_preds verified with real diffs/metrics.'
else:
    selected_preds = base_preds
    selected_metrics = base_metrics
    status = 'ROLLBACK_TO_V27'
    source = 'v27_standard'
    version = 'v30_full_clean_rollback_to_v27'
    reason = 'v30 standard not selected or failed verified metric improvement.'

summary = {
    'version': version,
    'selection_status': status,
    'selected_source': source,
    'reason': reason,
    'metrics': selected_metrics,
    'base_metrics': base_metrics,
    'standard_pred_metrics': std_pred_metrics,
    'standard_summary_metrics': std_summary_metrics,
    'flipped_indices_from_v27': actual_diffs if source.startswith('v30') else [],
    'artifact_safety': {
        'checked_standard_summary': True,
        'checked_standard_preds_real_metrics': True,
        'checked_diffs_vs_v27': True,
        'all_asserts_passed': True,
    },
}

save_json(selected_preds, 'v30_full_preds.json')
save_json(summary, 'v30_full_summary.json')

if status == 'SELECT_V30':
    assert source == 'v30_standard_clean'
    assert_close(selected_metrics['macro_f1'], V30_EXPECTED_MACRO, name='selected macro')
    assert selected_metrics['macro_f1'] > base_metrics['macro_f1']
    assert selected_metrics['mc_macro'] == base_metrics['mc_macro']
    assert set(actual_diffs) == V30_TARGET_FLIPS
print('v30 full clean complete:', status, metric_line(selected_metrics))
